# 📓 Exemplo 09: A janela móvel

**Missão 8** · [![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/change_climate/blob/main/code/notebooks_exemplo/09_janela_movel.ipynb)

> **Pergunta:** quando as folgas começaram a diminuir? A classe oscilou?

### 🧭 Como usar este notebook
1. 💾 **Faça uma cópia:** menu **Arquivo → Salvar uma cópia no Drive**. Mova a cópia para a pasta
   `mudanca_climatica_br/notebooks/`. Assim você pode mudar tudo sem estragar o original.
2. ▶️ **Rode uma célula por vez** (botão ▶ ou **Shift + Enter**) e **leia os comentários** (as linhas com `#`).
3. 🧪 Os exemplos usam **Santa Helena-PR** e **Curitiba-PR**.
4. ✍️ No fim, a seção **Sua vez** pede para você **replicar o método para as capitais do Brasil**.

In [ ]:
# Instala as bibliotecas que não vêm no Colab (só precisa rodar uma vez por sessão)
!pip install agrometeorologiapy pymannkendall

In [ ]:
# Endereço da pasta de dados do projeto no GitHub (arquivos "raw" = texto puro).
# O pandas consegue ler um CSV direto de um endereço da internet.
URL_DADOS = 'https://raw.githubusercontent.com/fcoliveira-utfpr/change_climate/main/code/dados'

In [ ]:
import pandas as pd

# Lê as séries mensais (1961-2025) direto do GitHub.
#   parse_dates=['data'] -> a coluna 'data' é entendida como data
#   index_col='data'     -> a coluna 'data' vira o índice (o "nome" de cada linha)
serie_sh = pd.read_csv(f'{URL_DADOS}/santa_helena_mensal.csv', parse_dates=['data'], index_col='data')
serie_ct = pd.read_csv(f'{URL_DADOS}/curitiba_mensal.csv', parse_dates=['data'], index_col='data')

LATITUDE_SH = -24.86
LATITUDE_CT = -25.43

serie_sh.head()

## 1. O `for` em 3 exemplos
O clima de cada ano é a normal dos **30 anos que terminam nele**: 1990 → 1961-1990, 1991 → 1962-1991 ...

In [ ]:
for ano in [1990, 1991, 1992]:
    print('O clima de', ano, 'usa os anos de', ano - 29, 'até', ano)

## 2. A janela móvel de Santa Helena, 1990 a 2025

In [ ]:
import agrometeorologiapy as amp

linhas = []                                   # lista vazia que vai receber uma linha por ano

for ano in range(1990, 2026):                 # 2026 não entra: o último ano é 2025
    # --- janela de 30 anos que termina em "ano" ---
    primeiro_ano = ano - 29
    janela = serie_sh.loc[str(primeiro_ano):str(ano)]

    # --- normal da janela ---
    normal = janela.groupby(janela.index.month).mean()
    temperatura = normal['tmed'].tolist()
    chuva = normal['chuva'].tolist()

    # --- Köppen e Thornthwaite ---
    koppen = amp.classificacao_koppen(temperatura, chuva, LATITUDE_SH)
    etp = amp.thornthwaite_mensal(pd.DataFrame({'T_media_C': temperatura}), lat=LATITUDE_SH)['ETP_mm_mes']
    thornthwaite = amp.classificacao_thornthwaite(chuva, etp, LATITUDE_SH)

    # --- guarda os resultados deste ano ---
    linha = {
        'ano': ano,
        'periodo': str(primeiro_ano) + '-' + str(ano),
        'koppen': koppen['classe'],
        'thornthwaite': thornthwaite['classe'],
        'folga_A': min(temperatura) - 18,
        'folga_a': max(temperatura) - 22,
        'folga_f': min(chuva) - 60,
        'Im': thornthwaite['Im'],
        'ETP_anual': thornthwaite['ETP_anual'],
    }
    linhas.append(linha)

# Fora do for: transforma a lista de linhas numa tabela
resultado_sh = pd.DataFrame(linhas).set_index('ano')
resultado_sh.round(2)

## 3. Gráficos das folgas no tempo

In [ ]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

eixos[0].plot(resultado_sh.index, resultado_sh['folga_A'], marker='o', color='tomato')
eixos[0].axhline(0, color='black', linestyle='--')          # o limiar
eixos[0].set_ylabel('Folga A/C (°C)')

eixos[1].plot(resultado_sh.index, resultado_sh['folga_a'], marker='o', color='orange')
eixos[1].axhline(0, color='black', linestyle='--')
eixos[1].set_ylabel('Folga a/b (°C)')

eixos[2].plot(resultado_sh.index, resultado_sh['ETP_anual'], marker='o', color='steelblue')
eixos[2].axhline(1140, color='black', linestyle='--')       # limite B'4 / A' do Thornthwaite
eixos[2].set_ylabel('ETP anual (mm)')
eixos[2].set_xlabel('Ano (fim da janela de 30 anos)')

for eixo in eixos:
    eixo.grid(alpha=0.3)
fig.suptitle('Clima de referência móvel: Santa Helena-PR')
plt.tight_layout()
plt.show()

## 4. A classe mudou? Oscilou?

In [ ]:
print(resultado_sh['koppen'].value_counts())

# .shift() coloca cada ano ao lado do ano anterior
classe_do_ano_anterior = resultado_sh['koppen'].shift()
mudou = resultado_sh['koppen'] != classe_do_ano_anterior
mudou = mudou.iloc[1:]                         # o primeiro ano não tem anterior

anos_que_mudaram = mudou[mudou == True].index
print('Anos em que a classe de Köppen mudou:')
print(resultado_sh.loc[anos_que_mudaram, ['periodo', 'koppen']])

> ⚠️ **Não aplique teste de tendência a estas curvas:** janelas vizinhas compartilham 29 anos. O ritmo é
> medido no Exemplo 11, na série **anual**.

> 🧠 **Curitiba:** copie a célula da seção 2, troque `serie_sh` → `serie_ct`, `LATITUDE_SH` → `LATITUDE_CT`
> e `resultado_sh` → `resultado_ct`. Olhe a folga **a/b** e a lista de anos em que a classe mudou: ela
> **oscila**?

## ✍️ Sua vez
1. Faça a janela móvel para **5 capitais** (copiando e adaptando a célula da seção 2).
2. Para cada uma, faça o gráfico da folga do critério mais próximo do zero.
3. Alguma oscila entre classes? Em que anos?
4. Repare quanto código você copiou. No Exemplo 10, isso vira **uma linha** por capital.

In [ ]:
# As capitais que VOCÊ extraiu na Missão 5 (exemplo 06) estão no seu Drive.
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mudanca_climatica_br'

# Tabela das 27 capitais (uf, capital, regiao, lat, lon), montada por você
capitais = pd.read_csv(f'{BASE}/dados/capitais_27.csv')

# Exemplo: lendo a série de uma capital (sigla em minúsculas)
uf = 'df'
serie_capital = pd.read_csv(f'{BASE}/dados/capitais/{uf}.csv', parse_dates=['data'], index_col='data')
serie_capital.head()